# Rate arithmetic

A flow rate is an expression tree. `tanh`, `abs`, `clip` and `**` are nodes
in that tree, so a triangular seed or a detection scale-up does not have to
hide inside a `Transform` callback. The same operators apply to a saved
`Trace`. A parameter-only piece such as `tanh(Param("se"))` is evaluated
with `eval_closed` and then scales the trace.


## Triangular seed

tb_macro's seeding pulse is `clip(h * (1 - abs(t - peak) / width), 0)`:
a triangle of height `h` and width `width`, zero outside that window.


In [ ]:
import numpy as np

from summer4 import (
    FlowModel,
    Param,
    Property,
    PropertyMap,
    Time,
    TransitionFlow,
    clip,
)

state = Property("state", ("S", "I"))
pmap = PropertyMap.from_property(state)
seed = clip(Param("height") * (1 - abs(Time() - Param("peak")) / Param("width")), 0)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("seed", state["S"], state["I"], seed, absolute=True))
compiled = model.compile()
params = {"height": 0.05, "peak": 20.0, "width": 8.0}
times = np.linspace(0.0, 40.0, 50)
y = np.array([1.0, 0.0])
got = np.array(
    [float(np.asarray(compiled.vector_field(float(t), y, params))[1]) for t in times]
)
expect = np.maximum(
    params["height"] * (1.0 - np.abs(times - params["peak"]) / params["width"]),
    0.0,
)
np.testing.assert_allclose(got, expect, rtol=1e-5, atol=1e-6)
assert got[0] == 0.0 and got[-1] == 0.0
assert got.max() > 0.9 * params["height"]
print(f"peak sample {got.max():.4f} (height {params['height']})")


## Tanh scale-up

Detection rises from `start` to `end`, with the midpoint at `inflection` and
steepness `shape`:
`start + (end - start) * (tanh(shape * (t - inflection)) + 1) / 2`.


In [ ]:
from summer4 import tanh

scale = Param("start") + (Param("end") - Param("start")) * (
    tanh(Param("shape") * (Time() - Param("inflection"))) + 1
) / 2
detect = FlowModel(pmap)
detect.add_flow(
    TransitionFlow("detect", state["S"], state["I"], scale, absolute=True)
)
compiled = detect.compile()
params = {"shape": 0.3, "inflection": 18.0, "start": 0.1, "end": 1.0}
got = np.array(
    [float(np.asarray(compiled.vector_field(float(t), y, params))[1]) for t in times]
)
expect = params["start"] + (params["end"] - params["start"]) * (
    np.tanh(params["shape"] * (times - params["inflection"])) + 1.0
) / 2.0
np.testing.assert_allclose(got, expect, rtol=1e-5, atol=1e-6)
assert got[0] < got[-1]
print(f"scale-up {got[0]:.3f} -> {got[-1]:.3f}")


## The same transform on a trace

`tanh(Param("se"))` is a rate expression. `eval_closed` evaluates the
parameter-only part against a parameter dict. The array scales a `Trace`
with the same operator the rate tree would use. The trace does not carry
parameters, so multiplying by the unevaluated expression is an error.


In [ ]:
from summer4 import Trace, eval_closed
from summer4.time import TimeAxis

expr = tanh(Param("se"))
factor = float(eval_closed(expr, {"se": 0.4}))
np.testing.assert_allclose(factor, np.tanh(0.4), rtol=1e-5, atol=1e-6)

constant = FlowModel(pmap)
constant.add_flow(
    TransitionFlow("detect", state["S"], state["I"], expr, absolute=True)
)
rate = float(np.asarray(constant.compile().vector_field(0.0, y, {"se": 0.4}))[1])
np.testing.assert_allclose(rate, factor, rtol=1e-5, atol=1e-6)

values = np.array([10.0, 20.0, 30.0, 40.0])
trace = Trace(
    times=TimeAxis(values=np.linspace(0.0, 3.0, 4), epoch=None, kind="explicit"),
    values=values,
    dims=("time",),
)
scaled = trace * factor
np.testing.assert_allclose(np.asarray(scaled.values), values * factor, rtol=1e-5)
try:
    trace * expr
except TypeError as exc:
    assert "eval_closed" in str(exc)
else:
    raise AssertionError("trace * unevaluated expr should fail")
print(f"sensitivity {factor:.4f}, scaled incidence {np.asarray(scaled.values)}")
